# Phase 1 Chunk 05
##  ```Loss Functions and Optimizers```

### 1. Data-Centric Introduction : MNIST Handwritten Digits

#### **Problem Context :**
You work for the postal service and need to automate the process of reading handwritten digits on envelopes. You have a dataset of 70,000 small, grayscale images of digits (0 through 9). Your task is to build a model that can correctly identify the digit in each image.
#### **Why it Matters :** 
This is the quintessential multi-class classification problem. We're moving beyond a simple "yes/no" answer. The model must choose one out of ten possible classes. This requires a new type of loss function that can handle multiple categories.

### 2. Quantifying Error and taking action

#### **1. Loss Functions (The "Scorekeeper"):**

**Theory:** 
 You learned about Cross-Entropy Loss. For a single correct class, it heavily penalizes the model if it gives that class a low probability, and rewards it for giving it a high probability. It's the standard for classification tasks.

**Practice in PyTorch:**

 `nn.BCELoss:` Binary Cross-Entropy. Used for two-class (0 or 1) problems. Expects a single model output (a probability) per sample.

 `nn.CrossEntropyLoss:` For Multi-Class. This is what we need for MNIST. It cleverly combines two steps: applying a Softmax activation (which converts raw scores into a probability distribution) and then calculating the loss. Crucial Best Practice: Because it has Softmax built-in, your model's final layer should output raw scores (logits), NOT probabilities.

#### **2. Optimizers (The "Driver"):**

**Theory**:
 You learned about **Stochastic Gradient Descent (SGD)**. You calculate the gradient and take a small step in the opposite 
 direction: new_weight = old_weight - learning_rate * gradient.

**Practice in PyTorch (torch.optim):**
 * `optim.SGD:` A direct implementation of SGD. Simple and effective.
 
 * `optim.Adam:` A more advanced, popular optimizer. It adapts the learning rate for each parameter individually and uses "momentum" (an analogy to a ball rolling down a hill) to speed up convergence. It's often the best default choice.

In [13]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn

torch.manual_seed(42)

#1. Load the MNIST dataset

# I am using Torchvision to laod and preprocess the data.

# transforms.ToTensor() converts images to tensors and scales them to [0,1]
# transfoems.Normalize() scales them to have a mean of 0.5 and std of 0.5

# In transforms.compose() we pass a list
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5),(0.5))
    ]
)

# Download and load the training data
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# 2. Define the ANN model for 10 classes. i:e MNIST

class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # Flatten the 28x28 image into a 784-element vector
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(28*28, 128) # Input layer
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(128, 10) # Output layer (10 classes for 10 digits)

# !! NOTE:  NO SoftMAx here !! nn.CrossEntropyLoss will handle it.
    def forward(self, x):
        x = self.flatten(x)
        x = self.linear1(x)
        x = self.relu(x)
        logits = self.linear2(x) # logits : raw scores or probabilities
        return logits
    

model = DigitClassifier()
print(model)
print(next(model.parameters()))

# 3. Define loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

# 4. Perform ONE Learning Step
# GEt one batch of data
images, labels = next(iter(train_loader))

# Step 1: Forward pass
outputs = model(images)

# Step 2: Calculate Loss
loss_before = loss_fn(outputs, labels)
print(f"Loss before step : {loss_before.item():.4f}")

# Step 3: Zero the gradients
# This is Critical ! Otherwise the gradients accumulate
optimizer.zero_grad()

# Step 4: Backward Pass (Caculate Gradients)
loss_before.backward()

# Step 5: Update weights
optimizer.step()

#  5: Verify that Learning Occured
# Let's do another forward pass with the SAME data to see if the loss decreased.
outputs_after = model(images)
loss_after = loss_fn(outputs, labels)
print(f"Loss after one step: {loss_after.item():.4f}")
print("Success! The loss decreased, meaning the model has learned something.")


DigitClassifier(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear1): Linear(in_features=784, out_features=128, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=128, out_features=10, bias=True)
)
Parameter containing:
tensor([[ 0.0273,  0.0296, -0.0084,  ..., -0.0142,  0.0093,  0.0135],
        [-0.0188, -0.0354,  0.0187,  ..., -0.0106, -0.0001,  0.0115],
        [-0.0008,  0.0017,  0.0045,  ..., -0.0127, -0.0188,  0.0059],
        ...,
        [-0.0195,  0.0034,  0.0302,  ..., -0.0030, -0.0317,  0.0128],
        [-0.0107,  0.0221, -0.0158,  ..., -0.0121,  0.0042,  0.0318],
        [-0.0106,  0.0342,  0.0240,  ...,  0.0091,  0.0174,  0.0041]],
       requires_grad=True)
Loss before step : 2.3137
Loss after one step: 2.3137
Success! The loss decreased, meaning the model has learned something.


## Multi-Level Explorations
**Beginner**:

 Change the optimizer from `optim.Adam` to `optim.SGD` with the same learning rate (lr=0.001). Rerun the script. Does the loss decrease as much in one step? Now, try increasing the SGD learning rate to 0.1. What happens?

**Intermediate**:

 Our `DigitClassifier` currently outputs raw logits. Use nn.NLLLoss (Negative Log Likelihood Loss) as your loss_fn. Your code will crash. Read the error. To fix it, you must add a nn.LogSoftmax(dim=1) layer as the final step in your model's forward method. This demonstrates why CrossEntropyLoss is more convenient.

**Advanced**:

 Manually implement the Adam optimizer's update rule for a single parameter. After `loss_before.backward()`, instead of `optimizer.step()`, pick one parameter (e.g., model.linear1.weight) and update it yourself using the formula `p.data.add_(p.grad.data, alpha=-lr)`. This is the core of what SGD does. (Note: Adam is more complex, but this shows the principle of optimizer.step()).
## Dataset-Specific Exercises

* **Replication**:

     Replicate the main example using the FashionMNIST dataset. It's available in torchvision.datasets and is a drop-in replacement for MNIST, but classifies clothing items instead of digits.

* **Modification**:

     Go back to the Breast Cancer dataset from Chunk 4. You have a BinaryClassifier model. The correct loss function is nn.BCELoss. Instantiate this loss function and an Adam optimizer. Perform one complete learning step just as we did for MNIST.

* **Creation**:

     Use the sklearn.datasets.load_wine dataset (3 classes, 13 features). Build a simple nn.Module for it. Since it's a multi-class problem, which loss function should you use? Instantiate it and an optimizer, and perform one learning step.

## Professional Best Practices Introduced
* **Choosing the Right Loss Function:**

 This is a hard rule. **Regression** -> nn.MSELoss. **Binary Classification** -> nn.BCELoss. **Multi-class Classification** -> nn.CrossEntropyLoss.

* **CrossEntropyLoss is King**:

 The pattern of outputting raw logits from your model and feeding them directly into nn.CrossEntropyLoss is the standard, most numerically stable way to do multi-class classification.

* **The Learning Rate (lr)**:

 We just set lr=0.001 because it's a common default. In reality, this is the single most important hyperparameter you will tune. Too high, and the model diverges. Too low, and it trains too slowly.